In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

In [ ]:
import json 

with open("/ir-project/data_retrieval/kochwiki_data/rezepte_parsed.json", "r") as f:
    recipes = json.load(f)

with open("/ir-project/data_retrieval/kochwiki_data/zutaten_parsed.json", "r") as f:
    ingredients = json.load(f)

data = recipes + ingredients

In [ ]:
from haystack import Document

docs = [Document(content=json.dumps(item)) for item in data]

In [ ]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(model="T-Systems-onsite/cross-en-de-roberta-sentence-transformer")
doc_embedder.warm_up()

In [ ]:
docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])

In [ ]:
from haystack.components.embedders import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(model="T-Systems-onsite/cross-en-de-roberta-sentence-transformer")

In [ ]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store)

In [ ]:
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage

template = [
    ChatMessage.from_user(
        """
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

prompt_builder = ChatPromptBuilder(template=template)

In [ ]:
import os
# set your Hugging Face API token as an environment variable here 

from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

generator = HuggingFaceAPIChatGenerator(
    api_type="serverless_inference_api",
    api_params={"model": "meta-llama/Meta-Llama-3-8B-Instruct"}
)
generator.warm_up()
print("Generator ready!")

In [ ]:
from haystack import Pipeline

text_embedder = SentenceTransformersTextEmbedder(model="T-Systems-onsite/cross-en-de-roberta-sentence-transformer")
retriever = InMemoryEmbeddingRetriever(document_store)
prompt_builder = ChatPromptBuilder(template=template)

basic_rag_pipeline = Pipeline()
basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", generator)


In [ ]:
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever", "prompt_builder")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

In [ ]:
question = "Kartoffelsuppe Rezept"

response = basic_rag_pipeline.run({"text_embedder": {"text": question}, "prompt_builder": {"question": question}})

print(response["llm"]["replies"][0].text)